# Advanced Ensemble Learning: Bagging vs. Boosting for Cardiovascular Risk Prediction
**Course / Lab Project**: Machine Learning Lab

This notebook implements, benchmarks, and optimizes:
1. **Bagging Paradigms** (Variance Reduction): Random Forest, Bagging Decision Trees, Extra Trees
2. **Boosting Paradigms** (Bias Reduction): XGBoost, HistGradientBoosting, Gradient Boosting, AdaBoost
3. **Hybrid Ensembles**: Soft-voting combination of top bagging and boosting models.

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    AdaBoostClassifier,
    VotingClassifier
)
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
)

## 1. Load Preprocessed Clinical Dataset

In [ ]:
df = pd.read_csv('../cleaned_cardio.csv') if pd.io.common.file_exists('../cleaned_cardio.csv') else pd.read_csv('cleaned_cardio.csv')
print(f'Loaded {df.shape[0]:,} records with {df.shape[1]} columns.')
X = df.drop(columns=['cardio'])
y = df['cardio']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train split: {len(X_train):,} | Test split: {len(X_test):,}')

## 2. Train and Benchmark Bagging vs. Boosting Models

In [ ]:
models = {
    'Random Forest (Bagging)': RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_split=5, random_state=42, n_jobs=4),
    'Bagging (Decision Tree)': BaggingClassifier(estimator=DecisionTreeClassifier(max_depth=10), n_estimators=100, random_state=42, n_jobs=4),
    'Extra Trees (Bagging)': ExtraTreesClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=4),
    'XGBoost (Boosting)': xgb.XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=4, subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', tree_method='hist', random_state=42, n_jobs=4),
    'Hist Gradient Boosting': HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_depth=5, l2_regularization=1.0, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.05, max_depth=4, random_state=42),
    'AdaBoost (Boosting)': AdaBoostClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = []
for name, clf in models.items():
    t0 = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - t0
    
    t1 = time.time()
    preds = clf.predict(X_test)
    probs = clf.predict_proba(X_test)[:, 1]
    infer_lat = ((time.time() - t1) * 1000) / len(X_test)
    
    results.append({
        'Model': name,
        'Accuracy': f'{accuracy_score(y_test, preds)*100:.2f}%',
        'Precision': f'{precision_score(y_test, preds)*100:.2f}%',
        'Recall': f'{recall_score(y_test, preds)*100:.2f}%',
        'F1 Score': f'{f1_score(y_test, preds)*100:.2f}%',
        'ROC-AUC': f'{roc_auc_score(y_test, probs)*100:.2f}%',
        'Train Time (s)': round(train_time, 2),
        'Latency (ms)': round(infer_lat, 4)
    })

results_df = pd.DataFrame(results).sort_values(by='ROC-AUC', ascending=False)
results_df

## 3. Hybrid Ensemble (Bagging + Boosting Soft Voting)

In [ ]:
hybrid_ensemble = VotingClassifier(
    estimators=[
        ('rf', models['Random Forest (Bagging)']),
        ('xgb', models['XGBoost (Boosting)']),
        ('hgb', models['Hist Gradient Boosting'])
    ],
    voting='soft'
)
hybrid_ensemble.fit(X_train, y_train)
hybrid_preds = hybrid_ensemble.predict(X_test)
hybrid_probs = hybrid_ensemble.predict_proba(X_test)[:, 1]
print(f'Hybrid Ensemble Accuracy: {accuracy_score(y_test, hybrid_preds)*100:.2f}%')
print(f'Hybrid Ensemble ROC-AUC:  {roc_auc_score(y_test, hybrid_probs)*100:.2f}%')